# DDF timeline: cumulative visits and cadence of the Deep Drilling Fields, in baseline survey v5.3

This notebook characterises the **Deep Drilling Fields (DDF)** of a simulated LSST survey: how many visits each DDF receives, when, in which bands, and with which cadence. The baseline v5.3 uses the *Ocean* DDF strategy, where each DDF alternates between shallow and deep seasons, so the **cumulative curves show a staircase pattern** rather than a straight line.

- creation date : 2026-10-09
- companion notebooks: `01_SurveyModes` (DDF positions and DDF fraction of the survey), `02_SurveyAreas` (DDF visit map)
- real-data counterpart (DP2, via Butler on the Rubin Science Platform): https://dp2.lsst.io/tutorials/notebook/301/notebook-301-3.html
- Ocean DDF strategy discussion: https://community.lsst.org/t/ddf-observations-under-the-ocean-observing-strategy/12370

## Overview and outputs

1. **Load the DDF visits** from the OpSim database (`scheduler_note` containing `DD`, RGES excluded) and assign each visit to a DDF.
2. **Summary table** per DDF: visits, fraction of the survey, nights, first/last visit, position, visits per band, exposure time.
3. **Figures**:
   - Figure 1: cumulative number of visits per DDF versus time;
   - Figure 2: cumulative visits per DDF and per band;
   - Figure 3: visits per month and per DDF (heat map);
   - Figure 4: nights on which each DDF is observed (timeline);
   - Figure 5: visits per DDF night and gaps between DDF nights;
   - Figure 6: cumulative fraction of the survey visits spent in DDFs.

**Input**: the OpSim database `baseline_v5.3.6_10yrs.db` (variable `opsdb`).

**Outputs**: the summary table (`.csv`) goes to `DATA04_DDF_timeline/`; all figures go to `FIG04_DDF_timeline/`.

## 0. Setup

In [ ]:
import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rubin_scheduler.utils import SURVEY_START_MJD
from rubin_sim.data import get_baseline

In [ ]:
# Output directories: tables go to DATA_DIR, all figures go to FIG_DIR
DATA_DIR = "DATA04_DDF_NVisitsVsTime"
FIG_DIR = "FIG04_DDF_NVisitsVsTime"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# OpSim database. Replace by an explicit path to analyse another run.
# opsdb = get_baseline()
opsdb = "/Users/dagoret/DATA/OpSim/baseline_v5.3.6_10yrs.db"
opsim_name = os.path.split(opsdb)[-1].replace(".db", "")

print(opsdb)

YEAR = 365.25  # days
BANDS = list("ugrizy")
# Standard Rubin band colours
BAND_COLORS = {
    "u": "#0c71ff",
    "g": "#49be61",
    "r": "#c61c00",
    "i": "#ffc200",
    "z": "#f341a2",
    "y": "#5d0000",
}

# The two EDFS pointings (EDFS_a, EDFS_b) are the same DDF: merge them if True
MERGE_EDFS = True

## 1. Load the DDF visits

All visits are read once (needed for the survey-wide fraction of Figure 6). DDF visits are the ones whose `scheduler_note` contains `DD`; the `RGES` visits (also tagged with `DD`) are excluded as in `02_SurveyAreas`. Time is expressed in **years since the survey start** (`SURVEY_START_MJD`).

The cell prints the list of `scheduler_note` values found, so you can check which DDFs are present. The DDF name is the text after `DD:`.

In [ ]:
query = (
    "select observationStartMJD, night, band, fieldRA, fieldDec, "
    "visitExposureTime, fiveSigmaDepth, scheduler_note from observations"
)
conn = sqlite3.connect(opsdb)
visits = pd.read_sql(query, conn)
conn.close()

visits["years"] = (visits["observationStartMJD"] - SURVEY_START_MJD) / YEAR
print(f"{len(visits)} visits in the survey, duration {visits['years'].max():.2f} years")

note = visits["scheduler_note"].fillna("")
is_ddf = note.str.contains("DD") & ~note.str.contains("RGES")
ddf = visits[is_ddf].copy()

# 'DD:COSMOS' -> 'COSMOS'
ddf["field"] = ddf["scheduler_note"].str.extract(r"DD:?\s*([A-Za-z0-9_]+)", expand=False)
ddf["field"] = ddf["field"].fillna(ddf["scheduler_note"])
if MERGE_EDFS:
    ddf["field"] = ddf["field"].str.replace(r"_[abAB]$", "", regex=True)

print(f"\n{len(ddf)} DDF visits ({100 * len(ddf) / len(visits):.2f}% of the survey)")
print("\nscheduler_note values selected as DDF:")
print(ddf["scheduler_note"].value_counts())

In [ ]:
# DDFs sorted by decreasing number of visits, and one colour per DDF
fields = ddf["field"].value_counts().index.tolist()
cmap = plt.get_cmap("tab10")
field_colors = {f: cmap(i % 10) for i, f in enumerate(fields)}
print(fields)

## 2. Summary table per DDF

For each DDF: number of visits, share of the whole survey, number of distinct nights, time of the first and last visit (years since survey start), median position (RA, Dec in degrees), total exposure time (hours) and number of visits per band. The last row sums all DDFs. The table is saved to `DATA04_DDF_timeline/`.

In [ ]:
def summary_row(d):
    row = {
        "N visits": len(d),
        "% of survey": 100 * len(d) / len(visits),
        "N nights": d["night"].nunique(),
        "first (yr)": d["years"].min(),
        "last (yr)": d["years"].max(),
        "RA (deg)": d["fieldRA"].median(),
        "Dec (deg)": d["fieldDec"].median(),
        "exposure (h)": d["visitExposureTime"].sum() / 3600,
    }
    for b in BANDS:
        row[b] = int((d["band"] == b).sum())
    return pd.Series(row)


summary = pd.DataFrame({f: summary_row(ddf[ddf["field"] == f]) for f in fields}).T
summary.loc["All DDF"] = summary_row(ddf)
summary = summary.round(2)
for col in ["N visits", "N nights"] + BANDS:
    summary[col] = summary[col].astype(int)

summary.to_csv(os.path.join(DATA_DIR, f"{opsim_name}_ddf_summary.csv"))
summary

## 3. Cumulative visits

### Figure 1 - Cumulative number of visits per DDF (`<run>_ddf_cumulative.png`)

One staircase curve per DDF: the number of visits accumulated since the survey start. Flat parts are seasons when the field is not observed (not visible or in an off phase); steep parts are the seasons with many visits (the deep seasons of the Ocean strategy). The legend gives the final number of visits of each DDF.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for f in fields:
    t = np.sort(ddf.loc[ddf["field"] == f, "years"].values)
    ax.step(t, np.arange(1, len(t) + 1), where="post", color=field_colors[f], lw=1.8, label=f"{f} ({len(t)})")
ax.set_xlabel("Time since survey start (years)")
ax.set_ylabel("Cumulative number of visits")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.grid(alpha=0.3)
ax.legend(title="DDF (total visits)")
ax.set_title(f"Cumulative DDF visits - {opsim_name}")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, f"{opsim_name}_ddf_cumulative.png"), dpi=150)

### Figure 2 - Cumulative visits per DDF and per band (`<run>_ddf_cumulative_per_band.png`)

One panel per DDF, one curve per band (standard Rubin colours). It shows how the visits are shared between filters and when each filter is used.

In [ ]:
ncols = 3
nrows = int(np.ceil(len(fields) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.6 * nrows), sharex=True, squeeze=False)
for ax, f in zip(axes.ravel(), fields):
    d = ddf[ddf["field"] == f]
    for b in BANDS:
        t = np.sort(d.loc[d["band"] == b, "years"].values)
        if len(t) == 0:
            continue
        ax.step(t, np.arange(1, len(t) + 1), where="post", color=BAND_COLORS[b], label=b)
    ax.set_title(f"{f} ({len(d)} visits)")
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.3)
    ax.set_xlabel("Years since survey start")
    ax.set_ylabel("Cumulative visits")
for ax in axes.ravel()[len(fields) :]:
    ax.set_visible(False)
axes.ravel()[0].legend(title="band")
fig.suptitle(f"Cumulative DDF visits per band - {opsim_name}")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, f"{opsim_name}_ddf_cumulative_per_band.png"), dpi=150)

## 4. Timing of the DDF observations

### Figure 3 - Visits per month and per DDF (`<run>_ddf_visits_per_month.png`)

Heat map: one row per DDF, one column per month (1/12 of a year) of the survey, colour = number of visits in that month. The seasonal visibility of each field and the shallow/deep alternation of the Ocean strategy appear as bright and dark blocks.

In [ ]:
bin_years = 1 / 12
edges = np.arange(0, np.ceil(visits["years"].max() / bin_years) + 1) * bin_years
counts = np.array([np.histogram(ddf.loc[ddf["field"] == f, "years"], bins=edges)[0] for f in fields])

fig, ax = plt.subplots(figsize=(11, 0.6 * len(fields) + 2))
im = ax.imshow(
    counts, aspect="auto", origin="upper", cmap="viridis", extent=[edges[0], edges[-1], len(fields), 0]
)
ax.set_yticks(np.arange(len(fields)) + 0.5)
ax.set_yticklabels(fields)
ax.set_xlabel("Time since survey start (years)")
fig.colorbar(im, ax=ax, label="Visits per month")
ax.set_title(f"DDF visits per month - {opsim_name}")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, f"{opsim_name}_ddf_visits_per_month.png"), dpi=150)

### Figure 4 - Nights with DDF observations (`<run>_ddf_nights.png`)

One row per DDF; each tick is one night on which the field received at least one visit. Dense groups of ticks are observing seasons, empty stretches are the periods when the field is not observed.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 0.6 * len(fields) + 2))
for i, f in enumerate(fields):
    t = np.sort(ddf.loc[ddf["field"] == f].drop_duplicates("night")["years"].values)
    ax.eventplot(t, lineoffsets=i, linelengths=0.8, colors=[field_colors[f]], linewidths=0.7)
ax.set_yticks(range(len(fields)))
ax.set_yticklabels(fields)
ax.invert_yaxis()
ax.set_xlim(left=0)
ax.set_xlabel("Time since survey start (years)")
ax.set_title(f"Nights with DDF visits - {opsim_name}")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, f"{opsim_name}_ddf_nights.png"), dpi=150)

### Figure 5 - Sequence length and gap between DDF nights (`<run>_ddf_cadence.png`)

- Left: number of visits taken on one night in one DDF (length of a DDF sequence). With the Ocean strategy, short and long sequences are expected for the shallow and deep phases.
- Right: gap in nights between two consecutive observing nights of the same DDF (log-log axes). The peak at a few nights is the in-season cadence; the long tail is the gap between seasons.

In [ ]:
per_night = ddf.groupby(["field", "night"]).size().rename("n").reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

bins_n = np.arange(0, per_night["n"].max() + 2) - 0.5
all_gaps = []
for f in fields:
    n = per_night.loc[per_night["field"] == f, "n"]
    axes[0].hist(n, bins=bins_n, histtype="step", color=field_colors[f], lw=1.6, label=f)
    gaps = np.diff(np.sort(per_night.loc[per_night["field"] == f, "night"].values))
    all_gaps.append(gaps)

gmax = max([g.max() for g in all_gaps if len(g) > 0] + [2])
bins_g = np.logspace(0, np.log10(gmax + 1), 40)
for f, gaps in zip(fields, all_gaps):
    if len(gaps) > 0:
        axes[1].hist(gaps, bins=bins_g, histtype="step", color=field_colors[f], lw=1.6, label=f)

axes[0].set_xlabel("Visits per night in one DDF")
axes[0].set_ylabel("Number of nights")
axes[0].grid(alpha=0.3)
axes[0].legend()
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("Gap between consecutive DDF nights (nights)")
axes[1].set_ylabel("Number of gaps")
axes[1].grid(alpha=0.3)
fig.suptitle(f"DDF cadence - {opsim_name}")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, f"{opsim_name}_ddf_cadence.png"), dpi=150)

## 5. DDF share of the survey

### Figure 6 - Cumulative fraction of survey visits in DDFs (`<run>_ddf_fraction.png`)

At each time, the number of DDF visits accumulated so far divided by the number of all visits accumulated so far, in percent. Thick black line: all DDFs; thin lines: each DDF. Its final value is the DDF share given in the summary table, so this shows whether the DDF time is spent evenly along the survey or concentrated at some epochs.

In [ ]:
t_all = np.sort(visits["years"].values)
grid = np.linspace(0.01, t_all.max(), 500)
n_all_cum = np.searchsorted(t_all, grid, side="right")

fig, ax = plt.subplots(figsize=(10, 5))
for f in fields:
    t = np.sort(ddf.loc[ddf["field"] == f, "years"].values)
    frac = 100 * np.searchsorted(t, grid, side="right") / np.maximum(n_all_cum, 1)
    ax.plot(grid, frac, color=field_colors[f], lw=1.2, label=f)
t_ddf = np.sort(ddf["years"].values)
frac_all = 100 * np.searchsorted(t_ddf, grid, side="right") / np.maximum(n_all_cum, 1)
ax.plot(grid, frac_all, color="k", lw=2.5, label="All DDF")

ax.set_xlabel("Time since survey start (years)")
ax.set_ylabel("Cumulative DDF visits / all visits (%)")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.grid(alpha=0.3)
ax.legend(ncol=2)
ax.set_title(f"Fraction of survey visits in DDFs - {opsim_name}")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, f"{opsim_name}_ddf_fraction.png"), dpi=150)